In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

# 加载数据
order_data = pd.read_csv("06 美团业务选题-优惠券投放提效分析-数据1-order_detail.csv").head(500000)
user_visit = pd.read_csv("06 美团业务选题-优惠券投放提效分析-数据3-user_visit_detail.csv").head(500000)
user_coupon = pd.read_csv("06 美团业务选题-优惠券投放提效分析-数据2-user_coupon_receive.csv").head(500000)

# 1. 数据合并与特征选择

In [ ]:
# 计算用户领取的优惠券总数
coupon_received = user_coupon.groupby('User_id')['Coupon_id'].count().reset_index(name='total_received')

# 计算用户使用的优惠券数量（状态为2表示已使用）
coupon_used = user_coupon[user_coupon['Coupon_status'] == 2].groupby('User_id')['Coupon_id'].count().reset_index(name='total_used')

# 合并特征
features = pd.merge(coupon_received, coupon_used, on='User_id', how='left').fillna(0)

# 计算核销率
features['redemption_rate'] = features['total_used'] / features['total_received']

# 合并订单数据，计算平均补贴占比
order_features = order_data.groupby('User_id').agg(
    avg_subsidy_ratio=('Reduce_amount', lambda x: np.mean(x / (x + order_data['Actual_pay'])))
)
features = pd.merge(features, order_features, on='User_id', how='left')

# 合并活跃数据，计算活跃天数
active_days = user_visit.groupby('User_id')['Visit_date'].nunique().reset_index(name='active_days')
features = pd.merge(features, active_days, on='User_id', how='left')

# 填充缺失值并标准化
features.fillna(0, inplace=True)
scaler = StandardScaler()
scaled_features = scaler.fit_transform(features[['total_received', 'total_used', 'redemption_rate', 'avg_subsidy_ratio', 'active_days']])


# 2. 训练孤立森林模型

In [ ]:
model = IsolationForest(n_estimators=100, contamination=0.05, random_state=42)  # 假设异常占比5%
model.fit(scaled_features)

# 3. 预测异常用户

In [ ]:
features['anomaly_score'] = model.decision_function(scaled_features)
features['is_anomaly'] = model.predict(scaled_features)
anomaly_users = features[features['is_anomaly'] == -1]['User_id'].tolist()
print(f"检测到异常用户数量：{len(anomaly_users)}")
print("示例异常用户ID：", anomaly_users[:5])

In [ ]:
features[features["User_id"] =='0002f37809f184420a7aa7227db685bb']

In [ ]:
order_data[order_data["User_id"]=='0053ef9e51d62b102064b441f74d395a']

In [ ]:
features.drop(columns="User_id").groupby("is_anomaly").mean()